# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pathakadithi/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. Two paper findings + my methodology questions

### Finding 1: The Anatomy of Growing Content

The paper reports that growing pages were younger on average than declining pages: about 185 days versus 228 days. It also reports that word count was nearly the same between the groups. The paper presents this as an observed pattern in its content dataset.

**My methodology question:**
Where does the "growing vs. declining" label come from, and exactly how is traffic direction defined? Since page age can be related to other factors such as search visibility, topic, and previous performance, I would want to know whether the comparison controls for these differences or is mainly a descriptive comparison. The paper itself notes that content age can confound model comparisons and that the study shows patterns rather than proof of cause and effect.

This does not mean the finding is wrong. It means I would interpret it as an observed relationship and avoid claiming that younger content itself causes growth.

### Finding 2: The Freshness Multiplier

The paper reports that recently refreshed content had stronger performance than content that had not been updated for longer periods. It defines freshness separately from age and uses recent performance comparisons to study the relationship between updating content and results.

**My methodology question:**
How is the "freshness" label or treatment defined, and what is the comparison group? In particular, if pages were refreshed because they were already performing differently or were selected for improvement, could that selection affect the measured result? I would want to know whether the validation design can distinguish the effect of a refresh from other changes happening at the same time.

I would therefore describe this as evidence of an observed relationship or directional signal rather than proof that refreshing a page will cause a specific amount of improvement.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

My Week-5 evaluation used a random stratified 80/20 split. This produced 95.28% accuracy, 44.32% precision, 99.95% recall, and 0.614 F1.

For this audit, I keep the same five features, target, Logistic Regression model, and evaluation metrics, but change the validation design to a client-grouped 80/20 split. Rows from the same `client_hash_id` are kept on the same side of the split, so the model is evaluated on clients that were not present in training.

I compare the original random-split result with the grouped result below. The grouped result is a more conservative check of how the model behaves when evaluated across unseen clients.

The comparison is treated as an observed validation result, not proof of generalization to all future clients.


In [4]:
from huggingface_hub import login

login()

In [5]:
from datasets import load_dataset
import numpy as np
import pandas as pd

# Load the same Week-5 dataset
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d"
)

df = dataset["train"].to_pandas()

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2414248 [00:00<?, ? examples/s]

Rows: 2414248
Columns: ['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


In [8]:
# Create CTR features
df["ctr_90d"] = np.where(
    df["impressions_90d"] > 0,
    df["clicks_90d"] / df["impressions_90d"],
    0
)

df["ctr_last30"] = np.where(
    df["impressions_last30"] > 0,
    df["clicks_last30"] / df["impressions_last30"],
    0
)

df["ctr_prev30"] = np.where(
    df["impressions_prev30"] > 0,
    df["clicks_prev30"] / df["impressions_prev30"],
    0
)

# Recreate Week-4 baseline
df["position_bucket"] = pd.cut(
    df["avg_position_90d"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

position_ctr = (
    df.groupby("position_bucket", observed=False)["ctr_90d"]
    .mean()
)

df["expected_ctr"] = (
    df["position_bucket"]
    .map(position_ctr)
    .astype(float)
)

df["ctr_gap"] = (
    df["expected_ctr"] - df["ctr_90d"]
).fillna(0)

df["recent_ctr_decline"] = (
    df["ctr_prev30"] - df["ctr_last30"]
).fillna(0)

ctr_gap_scale = df["ctr_gap"].quantile(0.95)
decline_scale = df["recent_ctr_decline"].quantile(0.95)

df["ctr_gap_score"] = (
    df["ctr_gap"] / ctr_gap_scale
).clip(0, 1)

df["decline_score"] = (
    df["recent_ctr_decline"] / decline_scale
).clip(0, 1)

df["baseline_score"] = (
    0.5 * df["ctr_gap_score"]
    + 0.5 * df["decline_score"]
)

df["baseline_action"] = (
    df["baseline_score"] > 0
).astype(int)

print("Features and target created.")
print("Rows:", len(df))
print("Positive labels:", df["baseline_action"].sum())

Features and target created.
Rows: 2414248
Positive labels: 90703


In [9]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import pandas as pd

# ------------------------------------------------------------
# Honest split: group by client_hash_id
# ------------------------------------------------------------

features = [
    "impressions_90d",
    "clicks_90d",
    "ctr_90d",
    "avg_position_90d",
    "avg_position_last30"
]

X = df[features]
y = df["baseline_action"]
groups = df["client_hash_id"]

group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    group_split.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]
y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

print("Training rows:", len(X_train_group))
print("Test rows:", len(X_test_group))

print("\nUnique training clients:",
      groups.iloc[train_idx].nunique())

print("Unique test clients:",
      groups.iloc[test_idx].nunique())

print("\nClient overlap:",
      len(
          set(groups.iloc[train_idx])
          & set(groups.iloc[test_idx])
      ))

# ------------------------------------------------------------
# Same Week-5 model
# ------------------------------------------------------------

group_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

group_model.fit(X_train_group, y_train_group)

y_pred_group = group_model.predict(X_test_group)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

group_accuracy = accuracy_score(y_test_group, y_pred_group)
group_precision = precision_score(y_test_group, y_pred_group, zero_division=0)
group_recall = recall_score(y_test_group, y_pred_group, zero_division=0)
group_f1 = f1_score(y_test_group, y_pred_group, zero_division=0)

print("\nGrouped validation results")
print("--------------------------")
print("Accuracy :", group_accuracy)
print("Precision:", group_precision)
print("Recall   :", group_recall)
print("F1 Score :", group_f1)

Training rows: 2164186
Test rows: 250062

Unique training clients: 41
Unique test clients: 11

Client overlap: 0

Grouped validation results
--------------------------
Accuracy : 0.9522118514608378
Precision: 0.4383638928067701
Recall   : 0.9995711835334476
F1 Score : 0.6094515981436696


## 2. My model under an honest split (before/after)

I first evaluated the Week-5 Logistic Regression model using a random stratified 80/20 split. I then re-ran the same model with the same five features and target, but used a client-grouped split so that no `client_hash_id` appeared in both training and test data.

| Validation design              | Accuracy | Precision | Recall |     F1 |
| ------------------------------ | -------: | --------: | -----: | -----: |
| Week-5 random stratified split |   0.9528 |    0.4432 | 0.9995 | 0.6141 |
| Week-6 client-grouped split    |   0.9522 |    0.4384 | 0.9996 | 0.6095 |

The grouped split had zero client overlap between training and test sets. The measured performance was slightly lower than the original random split: accuracy decreased from 0.9528 to 0.9522 and F1 decreased from 0.6141 to 0.6095. Recall remained approximately 1.00 in both evaluations.

This comparison suggests that the model's measured performance was broadly similar under the grouped validation design, although the grouped result is a more conservative estimate because the model is evaluated on clients that were not present during training. These results are validation observations and do not establish performance on all future clients.


In [10]:
comparison_table = pd.DataFrame({
    "Validation": [
        "Week-5 random stratified split",
        "Week-6 client-grouped split"
    ],
    "Accuracy": [
        0.9528031479755618,
        group_accuracy
    ],
    "Precision": [
        0.44319515056707076,
        group_precision
    ],
    "Recall": [
        0.9995038862245742,
        group_recall
    ],
    "F1": [
        0.6140924254483261,
        group_f1
    ]
})

comparison_table

,Validation,Accuracy,Precision,Recall,F1
0,Week-5 random stratified split,0.952803,0.443195,0.999504,0.614092
1,Week-6 client-grouped split,0.952212,0.438364,0.999571,0.609452


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I audited the five features used by the Week-5 model for information that would not be appropriate at prediction time or that overlaps with the target construction.

| Feature               | Available before scoring?                                         | Leakage concern          | Audit result                                        |
| --------------------- | ----------------------------------------------------------------- | ------------------------ | --------------------------------------------------- |
| `impressions_90d`     | Yes, if the scoring window is complete                            | Low                      | Historical performance feature                      |
| `clicks_90d`          | Yes, if the scoring window is complete                            | Low                      | Historical performance feature                      |
| `ctr_90d`             | Yes, because it is calculated from `clicks_90d / impressions_90d` | **Yes — target overlap** | Used indirectly in the baseline target construction |
| `avg_position_90d`    | Yes, if the 90-day window is complete before scoring              | Low                      | Historical ranking feature                          |
| `avg_position_last30` | Yes, if the last-30-day window ends before scoring                | Low                      | Recent historical ranking feature                   |

The main concern is not that these features contain future data by themselves. The main concern is that the target `baseline_action` was constructed from the same historical CTR information used by the model. In particular, `ctr_90d` contributes to `ctr_gap`, and `ctr_last30` and `ctr_prev30` contribute to `recent_ctr_decline`, which together create `baseline_action`.

Therefore, the Week-5 Logistic Regression result should not be interpreted as evidence that the model independently predicts real-world content-refresh outcomes. It measures how well the model reproduces the constructed Week-4 baseline action under the evaluated split.

The client-grouped validation removes overlap between clients, but it does not remove this target-construction overlap. This is an important limitation of the current evaluation and should be considered when interpreting the measured precision, recall, accuracy, and F1.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite
## 4. Claim rewrite

**Original claim:**

> The Logistic Regression model accurately predicts which content needs to be refreshed.

**Rewritten claim:**

> On the evaluated dataset, the Logistic Regression model measured 95.22% accuracy and 0.6095 F1 under client-grouped validation, compared with 95.28% accuracy and 0.6141 F1 under the original random split. These results provide a directional signal that may support content-refresh decision-making, but they do not establish that the model predicts real-world refresh success.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

- [x] Every section above is filled — markdown thinking and supporting code where required.
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, URLs, or private queries anywhere.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [ ] Committed to my repo under `work/notebooks/w06_validation_audit.ipynb` — then submit my repo URL on the card.